# Modeling: Spaceship Titanic

Author: Finian O'Neill
Purpose: Turn the EDA findings (`eda.ipynb`, section 15) into a validated model and a submission.

**Contents**
1. Setup
2. Preprocessing: imputation rules and features
3. Validation setup
4. Baseline: logistic regression
5. CatBoost
6. Ablation: do the EDA features help?
7. Feature importance
8. Results
9. Submission
10. Notes & next steps

Preprocessing lives in `app/preprocessing.py` and the CV and model helpers in `app/modeling.py`, so this notebook only records experiments.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from app import preprocessing as pp
from app import modeling as mdl

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
plt.style.use('fivethirtyeight')

## 2. Preprocessing: imputation rules and features
The deterministic rules from the EDA run first, then a statistical fallback using train-only statistics. The log shows how many values each step filled.

In [ ]:
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

full_df, imputation_log = pp.preprocess(train_df, test_df)
imputation_log

In [ ]:
# The EDA rules still hold after imputation
print('CryoSleep passengers with spend:', int((full_df.CryoSleep & (full_df.TotalSpending > 0)).sum()))
print('Children (<13) with spend:      ', int(((full_df.Age < 13) & (full_df.TotalSpending > 0)).sum()))

X, y, groups, X_test, test_ids = pp.split_xy(full_df)
print(f'\nX: {X.shape}, X_test: {X_test.shape}')
print(f'Categorical ({len(pp.CATEGORICAL_FEATURES)}): {pp.CATEGORICAL_FEATURES}')
print(f'Numeric ({len(pp.NUMERIC_FEATURES)}): {pp.NUMERIC_FEATURES}')

## 3. Validation setup
`StratifiedGroupKFold` (5 folds, fixed seed) on `GroupId`, so groupmates never land on both sides of a split. Every experiment uses the same splits, so their scores can be compared directly.

In [ ]:
splits = mdl.cv_splits(y, groups)
for i, (tr_idx, va_idx) in enumerate(splits):
    shared = set(groups.iloc[tr_idx]) & set(groups.iloc[va_idx])
    print(f'Fold {i}: train {len(tr_idx)}, valid {len(va_idx)}, '
          f'valid target rate {y.iloc[va_idx].mean():.3f}, shared groups {len(shared)}')

results = []

## 4. Baseline: logistic regression
One-hot categoricals and scaled numerics. Because it's linear, the gap between it and the tree models shows how much the interactions are worth.

In [ ]:
results.append(mdl.cross_validate(
    mdl.logreg_fit_predict(pp.CATEGORICAL_FEATURES), X, y, splits, name='LogReg (all features)'))

## 5. CatBoost
Native categorical handling, default-ish parameters (1,000 iterations, lr 0.03, depth 6). No tuning yet.

In [ ]:
cb_all = mdl.cross_validate(
    mdl.catboost_fit_predict(pp.CATEGORICAL_FEATURES), X, y, splits, name='CatBoost (all features)')
results.append(cb_all)

## 6. Ablation: do the EDA features help?
Drop each engineered feature (or group) and re-run CatBoost on the same splits. If the score drops, the feature helps.

Each fold's accuracy has a standard deviation of about 0.012 across folds, so differences in the mean of a few thousandths are within noise *on their own*. Because every run uses the **same splits**, the fold-by-fold comparison is more sensitive: a feature that lowers accuracy in 5 of 5 folds when dropped is very likely real.

In [ ]:
ablations = {
    'LuxuryShare': ['LuxuryShare'],
    'CryoSleep_Deck': ['CryoSleep_Deck'],
    'Deck_Side': ['Deck_Side'],
    'Route': ['Route'],
    'all 4 interactions': ['LuxuryShare', 'CryoSleep_Deck', 'Deck_Side', 'Route'],
    'spend aggregates': ['log_TotalSpending', 'log_LuxurySpending', 'log_BasicSpending',
                         'HasSpent', 'NumAmenitiesUsed', 'LuxuryShare'],
}
ablation_results = []
for label, drop in ablations.items():
    cols = [c for c in X.columns if c not in drop]
    r = mdl.cross_validate(mdl.catboost_fit_predict(pp.CATEGORICAL_FEATURES), X[cols], y, splits,
                           name=f'CatBoost − {label}')
    r['delta'] = r['cv_mean'] - cb_all['cv_mean']
    r['folds_lower'] = int((np.array(r['folds']) < np.array(cb_all['folds'])).sum())
    ablation_results.append(r)

pd.DataFrame([{'Dropped': r['name'].replace('CatBoost − ', ''), 'CV mean': r['cv_mean'],
               'Δ vs all features': r['delta'], 'Folds lower (of 5)': r['folds_lower']}
              for r in ablation_results]).set_index('Dropped').round(4)

## 7. Feature importance
CatBoost `PredictionValuesChange` importance, from a model fit on the full training set.

In [ ]:
cb_full = mdl.catboost_fit_predict(pp.CATEGORICAL_FEATURES)
test_proba_cb = cb_full(X, y, X_test)
importance = (pd.Series(cb_full.last_model.get_feature_importance(), index=X.columns)
                .sort_values(ascending=True))
plt.figure(figsize=(9, 8))
importance.plot.barh(color='#1b9e77')
plt.title('CatBoost feature importance (full train)', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Results

In [ ]:
table = mdl.results_table(results + ablation_results)
table.sort_values('CV mean', ascending=False)

In [ ]:
# Out-of-fold accuracy by key segment, to see where the model is weakest
oof_pred = (cb_all['oof'] >= 0.5).astype(int)
seg = full_df[full_df.Dataset == 'train'].reset_index(drop=True).assign(correct=(oof_pred == y.values))
seg['Segment'] = np.select(
    [seg.CryoSleep, seg.HasSpent == 1],
    ['CryoSleep', 'Awake, spender'], default='Awake, non-spender')
seg.groupby('Segment').agg(N=('correct', 'size'), OOF_accuracy=('correct', 'mean'),
                           Transported_rate=('Transported', 'mean')).round(3)

## 9. Submission
The CatBoost model fit on the full training set (section 7), using a 0.5 threshold. Files go to `submissions/`, which git ignores.

In [ ]:
sub_path = mdl.write_submission(test_ids, test_proba_cb, 'catboost_all', cb_all['cv_mean'])
pd.read_csv(sub_path).head()

To submit (from the repo root, with the env active):
```bash
kaggle competitions submit -c spaceship-titanic -f <path above> -m "CatBoost all features, CV <score>"
```
Then record the public leaderboard score in section 10.

## 10. Notes & next steps

> `model.ipynb` outputs are stripped on commit (nbstripout), so the key numbers are recorded here.

**Results (5-fold StratifiedGroupKFold on GroupId, seed 42)**

| Experiment | CV accuracy | Public LB |
|---|---|---|
| LogReg, all 24 features | 0.7990 ± 0.0161 | – |
| **CatBoost, all 24 features** | **0.8164 ± 0.0121** | *(fill in after submitting)* |

**What we learned**
- **Trees beat the linear model by about 1.7 points.** The interactions matter, and CatBoost's native categorical handling makes it a strong single-model baseline.
- **The EDA features help.** Dropping all four interaction features costs 0.45 points and lowers accuracy in 5 of 5 folds. `Deck_Side` alone costs 0.41 points (5 of 5 folds). `LuxuryShare`, `CryoSleep_Deck` and `Route` each cost about 0.2 points (4 of 5 folds). They're small one at a time but consistent, so **keep all of them**.
- **`LuxuryShare` is CatBoost's most important feature** by a wide margin, followed by `Cabin_num`, `HomePlanet` and `Deck`. That matches the EDA. `VIP_int`, `IsSolo` and `HasSpent` contribute almost nothing (`HasSpent` overlaps with `LuxuryShare` and the log spending columns).
- **The weakest segment is awake non-spenders** (532 passengers, 71.8% out-of-fold accuracy, mostly children). Sleepers are predicted at 81.6% accuracy, which is about the most possible given that sleepers on decks E and G are only 65% transported.

**Next steps**
1. **Submit** `submissions/catboost_all_cv0.8164.csv` and record the public leaderboard score above. If it's more than about 1 point off CV, check the validation setup before going further.
2. **LightGBM and XGBoost** on the same splits, then **average** the out-of-fold probabilities (the out-of-fold predictions are already stored in each `results` entry).
3. **Tune the threshold** on the out-of-fold predictions. The split is roughly 50/50, so expect small gains.
4. **Optuna** on CatBoost (depth, l2_leaf_reg, learning_rate and iterations, with early stopping inside each fold), about 50 trials. Do this only after the feature set is fixed.
5. **Features to try:** whether groupmates were asleep (`GroupCryoShare`), and finer age bands for children (the weakest segment). Drop `VIP_int` and `IsSolo` if an ablation confirms they're noise.